# Tarea 1 — Creación y operaciones básicas con PySpark

**Materia:** Datos Masivos  
**Alumno:** [Sergio Cortes Cepeda]  


---

## Dataset

**Nombre del dataset:** Flight List (enero 2019)  
**Fuente:** OpenFlights / Zenodo  
**Formato:** CSV  

Este conjunto de datos contiene información sobre vuelos comerciales, incluyendo fechas, aeropuertos de origen y destino, aerolíneas y tiempos de retraso.

---

##  Justificación del dataset

El dataset fue seleccionado debido a su volumen y estructura, lo cual lo hace adecuado para el uso de herramientas de procesamiento distribuido como Apache Spark.  
Además, permite realizar operaciones de filtrado, agregación y análisis descriptivo sobre variables relevantes para el análisis de datos masivos, como retrasos en vuelos y patrones de operación aérea.

---


In [17]:
import sys
sys.version


'3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]'

In [18]:
import pyspark
pyspark.__version__

'4.1.1'

In [19]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Tarea1_DatosMasivos_PySpark")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)

Spark version: 4.1.1


In [20]:
df = (spark.read
      .option("header", "true")
      .option("inferSchema", "true")
      .csv("flightlist_20190101_20190131.csv.gz")
)

df.printSchema()
df.show(5)

root
 |-- callsign: string (nullable = true)
 |-- number: string (nullable = true)
 |-- icao24: string (nullable = true)
 |-- registration: string (nullable = true)
 |-- typecode: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- firstseen: timestamp (nullable = true)
 |-- lastseen: timestamp (nullable = true)
 |-- day: timestamp (nullable = true)
 |-- latitude_1: double (nullable = true)
 |-- longitude_1: double (nullable = true)
 |-- altitude_1: double (nullable = true)
 |-- latitude_2: double (nullable = true)
 |-- longitude_2: double (nullable = true)
 |-- altitude_2: double (nullable = true)

+--------+------+------+------------+--------+------+-----------+-------------------+-------------------+-------------------+------------------+------------------+----------+------------------+------------------+----------+
|callsign|number|icao24|registration|typecode|origin|destination|          firstseen|           lastseen|     

In [22]:
df.describe().show()

+-------+--------+--------+--------+------------------+--------+-------+-----------+------------------+-------------------+------------------+------------------+-------------------+------------------+
|summary|callsign|  number|  icao24|      registration|typecode| origin|destination|        latitude_1|        longitude_1|        altitude_1|        latitude_2|        longitude_2|        altitude_2|
+-------+--------+--------+--------+------------------+--------+-------+-----------+------------------+-------------------+------------------+------------------+-------------------+------------------+
|  count| 2145469|   56934| 2145469|           1903420| 1692173|1491806|    1594704|           2145469|            2145469|           2145469|           2145418|            2145418|           2085176|
|   mean|Infinity|Infinity|Infinity|102069.41650059784|    NULL|   NULL|       NULL|30.689692558922086|-10.589362286834586|2373.8574721097502|30.713143384738583|-10.876396268794199|2427.9579254416

In [23]:
df.filter(df.origin == "LEMD").show(5)

+--------+------+------+------------+--------+------+-----------+-------------------+-------------------+-------------------+------------------+-------------------+----------+------------------+-------------------+----------+
|callsign|number|icao24|registration|typecode|origin|destination|          firstseen|           lastseen|                day|        latitude_1|        longitude_1|altitude_1|        latitude_2|        longitude_2|altitude_2|
+--------+------+------+------------+--------+------+-----------+-------------------+-------------------+-------------------+------------------+-------------------+----------+------------------+-------------------+----------+
|  AEA040|  NULL|34444e|      EC-LVL|    A332|  LEMD|       LEMD|2018-12-30 19:07:21|2018-12-31 21:32:59|2018-12-31 18:00:00|40.534755900754774| -3.575425581498564|     609.6|40.475727663201795| -3.538346724076689|    411.48|
| IBE6346|  NULL|345109|      EC-MKI|    A332|  LEMD|       LEMD|2018-12-31 05:07:49|2019-01-01 

In [21]:
df.filter(df.registration.isNull()).show(5)

+--------+------+------+------------+--------+------+-----------+-------------------+-------------------+-------------------+------------------+------------------+----------+-----------------+-------------------+-----------------+
|callsign|number|icao24|registration|typecode|origin|destination|          firstseen|           lastseen|                day|        latitude_1|       longitude_1|altitude_1|       latitude_2|        longitude_2|       altitude_2|
+--------+------+------+------------+--------+------+-----------+-------------------+-------------------+-------------------+------------------+------------------+----------+-----------------+-------------------+-----------------+
|   HVN19|  NULL|888152|        NULL|    NULL|  YMML|       LFPG|2018-12-30 18:43:16|2018-12-31 22:56:29|2018-12-31 18:00:00|-37.65948486328125|144.80442128282908|     304.8|48.99531555175781|  2.610802283653846|           -53.34|
|  CCA839|  NULL|780ad1|        NULL|    NULL|  YMML|       LEBL|2018-12-30 

In [24]:
from pyspark.sql.functions import col

df.select(
    col("altitude_1"),
    (col("altitude_1") + 100).alias("altitude_1_mas_100")
).show(5)

+----------+------------------+
|altitude_1|altitude_1_mas_100|
+----------+------------------+
|     304.8|             404.8|
|     304.8|             404.8|
|       0.0|             100.0|
|     609.6|             709.6|
|       0.0|             100.0|
+----------+------------------+
only showing top 5 rows
